# Bias Correction 

## Load Packages, libraries, etc.

In [4]:
##Import different packages
import datetime as dt
import importlib
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import copy as cp
import pandas as pd
import sys
import warnings
import math
import os
import cftime
from glob import glob
from scipy import sparse
import cartopy.crs as ccrs
from timeit import default_timer as timer
from os import mkdir, remove, rmdir

from climada.engine import Impact, ImpactCalc
from climada.entity import ImpactFunc,ImpactFuncSet

import warnings
warnings.filterwarnings('ignore') 
warnings.simplefilter('ignore')
import logging
from climada.util.config import LOGGER
LOGGER.setLevel(logging.ERROR) 

##Load self-written, custom function definitions
from SL_bias_corr import *
from constants import *
from functions_projUncWS import *
idx = pd.IndexSlice

In [5]:
#Read all ERA5 hazard documents
#Load ERA5 data
ds2000_2003 = xr.open_dataset('/Users/anne-sophiesimons/Documents/Data thesis/Data België/wind_gust_00_03')
ds2004_2007 = xr.open_dataset('/Users/anne-sophiesimons/Documents/Data thesis/Data België/wind_gust_04_07')
ds2008_2011 = xr.open_dataset('/Users/anne-sophiesimons/Documents/Data thesis/Data België/wind_gust_08_11')
ds2012_2014 = xr.open_dataset('/Users/anne-sophiesimons/Documents/Data thesis/Data België/wind_gust_12_14')

ds_combined = xr.concat([ds2000_2003, ds2004_2007, ds2008_2011, ds2012_2014], dim="valid_time")

#Calculate the daily maxima WG10 from ERA5 at each coordinate
df_combined = ds_combined.to_dataframe().reset_index()
df_combined


df_daily_max = df_combined.groupby(
    [df_combined['valid_time'].dt.date, df_combined['latitude'], df_combined['longitude']]
)['fg10'].max().reset_index()

df_daily_max

#'valid_time' must be a datetime
df_daily_max['valid_time'] = pd.to_datetime(df_daily_max['valid_time'])

#DataFrame to xarray Dataset
ds_daily_max = xr.Dataset.from_dataframe(df_daily_max.set_index(['valid_time', 'latitude', 'longitude']))

In [62]:
#Load CMIP6 data
bncmip6_historical = xr.open_dataset('/Users/anne-sophiesimons/Documents/Data thesis/CMIP6/CNRM-CM6-1/sfcWindmax_day_CNRM-CM6-1_historical_r1i1p1f2_gr_19500101-20141231.nc')

#Boundaries of Belgium
lat_min, lat_max = 49.5, 51.6
lon_min, lon_max = 2.5, 6.5

#Select the lat and lon of Belgium
ds_belgium_historical = bncmip6_historical.sel(
    lat=slice(lat_min, lat_max),
    lon=slice(lon_min, lon_max)
)

winter_months = [1, 2, 3, 10, 11, 12]

cmip6_hist_winter = ds_belgium_historical.sel(
    time=ds_belgium_historical.time.dt.month.isin(winter_months)
)

#Filter time between 2000-01-01 and 2014-12-31
start_date = np.datetime64('2000-01-01')
end_date   = np.datetime64('2015-01-01')

ds_belgium_historical = cmip6_hist_winter.sel(
    time=slice(start_date, end_date)
)

#Set to dataframe
df = ds_belgium_historical.to_dataframe().reset_index()

import xarray as xr
import numpy as np
from timeit import default_timer as timer

#Open datasets
bncmip6_historical = ds_belgium_historical
fnera5 = ds_daily_max

#Parameters
savencdf = True
pastname = "historical"  # naam van je historische data in bncmip6
modname = "GCM_model"    # geef hier je modelnaam op
data_folder = "/Users/axelleclaes/Desktop/wetransfer_thesis_2025-11-07_0852/bias_corrected"

#ERA5 data
era_da = fnera5['fg10']
era_df = era_da.to_dataframe().reset_index()

#lat en long van bncmip6 dataset nodig want we gaan daarop moeten correcten dus alles komt op deze grids terecht
latout = bncmip6_historical.lat
lonout = bncmip6_historical.lon
#Alle tijdstappenin het cmip6 bestand
daysid = bncmip6_historical.time
memid = bncmip6_historical.member if "member" in bncmip6_historical.dims else [0]  # check of er leden zijn
memid
daysid
latout
lonout

<xarray.DataArray 'lon' (lon: 3)> Size: 24B
array([2.8125 , 4.21875, 5.625  ])
Coordinates:
  * lon      (lon) float64 24B 2.812 4.219 5.625
    height   float64 8B 10.0
Attributes:
    axis:           X
    standard_name:  longitude
    long_name:      Longitude
    units:          degrees_east

In [63]:
#Load CMIP6 data
bncmip6_historical_all = xr.open_dataset('/Users/anne-sophiesimons/Documents/Data thesis/CMIP6/CNRM-CM6-1/sfcWindmax_day_CNRM-CM6-1_historical_r1i1p1f2_gr_19500101-20141231.nc')
import xarray as xr

#Boundaries of Belgium
lat_min, lat_max = 49.5, 51.6
lon_min, lon_max = 2.5, 6.5

#Select only lon and lat of Belgium
ds_belgium_hist_all = bncmip6_historical_all.sel(
    lat=slice(lat_min, lat_max),
    lon=slice(lon_min, lon_max)
)

bncmip6_hist_all = ds_belgium_hist_all

winter_months = [1, 2, 3, 10, 11, 12]

months_hist_all = ds_belgium_hist_all.sel(
    time=ds_belgium_hist_all.time.dt.month.isin(winter_months)
)

daysid_hist_all = pd.to_datetime(months_hist_all['time'].values)

da_corrected_hist_all = xr.DataArray(
    np.zeros([len(daysid_hist_all), latout.size, lonout.size, len(memid)]),
    coords=[daysid_hist_all, latout, lonout, memid],
    dims=['time', 'lat', 'lon', 'member'],
    name=pastname
)


df_corrected_hist_all = da_corrected_hist_all.to_dataframe().reset_index()
df_corrected_hist_all

,time,lat,lon,member,historical
0,1950-01-01 12:00:00,49.727115,2.81250,0,0.0
1,1950-01-01 12:00:00,49.727115,4.21875,0,0.0
2,1950-01-01 12:00:00,49.727115,5.62500,0,0.0
3,1950-01-01 12:00:00,51.127867,2.81250,0,0.0
4,1950-01-01 12:00:00,51.127867,4.21875,0,0.0
...,...,...,...,...,...
71071,2014-12-31 12:00:00,49.727115,4.21875,0,0.0
71072,2014-12-31 12:00:00,49.727115,5.62500,0,0.0
71073,2014-12-31 12:00:00,51.127867,2.81250,0,0.0
71074,2014-12-31 12:00:00,51.127867,4.21875,0,0.0


In [46]:
#ERA5 data moet men lineair gaan interpoleren omdat men bij ERA5 dataset verschillende grid cells heeft vergeleken met CMIP6 modellen
era_qt_rg = era_da.interp(latitude =latout, longitude =lonout, method='linear', kwargs={"fill_value": 'extrapolate'})
#ERA5 nu nog enkel latitude 50.62 en longitude 2.812 en 4.688
era_qt_rg

# Start bias correctie
start_time = timer()

for ilat in latout:
    for ilon in lonout:
        #Selecteer GCM data op deze gridpoint
        ncdfw_point = bncmip6_historical.sel(lat=ilat, lon=ilon)
        ncdfw_df = bncmip6_historical.sel(lat=ilat, lon=ilon).to_dataframe().drop(['lat','lon'], axis=1).unstack()

        #Observaties van ERA5 op deze gridpoint
        dat_obs = era_qt_rg.sel(lat=ilat, lon=ilon, method = 'nearest').to_dataframe().drop(['lat','lon'], axis=1)['fg10']
        dat_mod = bncmip6_historical.sel(lat=ilat, lon=ilon, method = 'nearest').to_dataframe().drop(['lat','lon'], axis=1)['sfcWindmax'].unstack()
        dat_mod_all = months_hist_all.sel(lat=ilat, lon=ilon, method = 'nearest').to_dataframe().drop(['lat','lon'], axis=1)['sfcWindmax'].unstack()
        #Maar 1 kolom kiezen, want er zijn 2 identieke kolommen
        dat_mod = dat_mod.iloc[:, 0]
        dat_mod_all = dat_mod_all.iloc[:, 0]
        dat_mod.index = dat_mod.index.normalize()  # ERA5 en cmip6 moeten dezelfde tijden bevatten. Dit is ook zo maar bij cmip6 staat er nog 12:00 bij en dat moet men wegdoen
        dat_mod_all.index = dat_mod_all.index.normalize() 

        #dat_mod.index = pd.to_datetime(daysid.values)
        # Bias correctie met enkelvoudige tijdreeks

        # Reindex op ERA5 tijd
        #Alleen de overlappende periode overhouden tussen ERA5 en CMIP6 (2000-2004)
        dat_mod = dat_mod.reindex(dat_obs.index).dropna()

        # Bias correctie
        dat_mod_corrected = bias_correct_time_series(dat_mod, dat_obs, dat_mod_all) 

        # Opslaan in DataArray
        ###Hier zit ergens een fout in ####    
        da_corrected_hist_all.loc[dict(lat=ilat, lon=ilon, member=0)] = dat_mod_corrected.values
        #da_corrected.loc[dict(lat=ilat, lon=ilon)] = dat_mod_corrected.values

        print(ilat)
        print(ilon)

end_time = timer()
print("Tijd voor bias correctie:", end_time - start_time)

<xarray.DataArray 'lat' ()> Size: 8B
array(49.72711456)
Coordinates:
    lat      float64 8B 49.73
    height   float64 8B 10.0
Attributes:
    axis:           Y
    standard_name:  latitude
    long_name:      Latitude
    units:          degrees_north
<xarray.DataArray 'lon' ()> Size: 8B
array(2.8125)
Coordinates:
    lon      float64 8B 2.812
    height   float64 8B 10.0
Attributes:
    axis:           X
    standard_name:  longitude
    long_name:      Longitude
    units:          degrees_east
<xarray.DataArray 'lat' ()> Size: 8B
array(49.72711456)
Coordinates:
    lat      float64 8B 49.73
    height   float64 8B 10.0
Attributes:
    axis:           Y
    standard_name:  latitude
    long_name:      Latitude
    units:          degrees_north
<xarray.DataArray 'lon' ()> Size: 8B
array(4.21875)
Coordinates:
    lon      float64 8B 4.219
    height   float64 8B 10.0
Attributes:
    axis:           X
    standard_name:  longitude
    long_name:      Longitude
    units:          degr

In [47]:
#De namen veranderen zodat de onderliggende functies gewoon kunnen blijven draaien
da_corrected_hist_all = da_corrected_hist_all.rename({"lat": "latitude", "lon": "longitude"})
da_corrected_hist_all = da_corrected_hist_all.rename({"time": "valid_time"})

df_corrected_historical_all = da_corrected_hist_all.to_dataframe().reset_index()

bias_corr_ds_historical_all = da_corrected_hist_all.to_dataset()

# Opslaan
bias_corr_ds_historical_all.to_netcdf('/Users/anne-sophiesimons/Documents/Data thesis/Bias Corrected data/België/bias_corr_historical_1')

In [48]:
bias_corr_historical_1 = xr.load_dataset('/Users/anne-sophiesimons/Documents/Data thesis/Bias Corrected data/België/bias_corr_historical_1')

In [49]:
bias_corr_historical_1

<xarray.Dataset> Size: 663kB
Dimensions:     (valid_time: 11846, latitude: 2, longitude: 3, member: 1)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 95kB 1950-01-01T12:00:00 ... 2014...
  * latitude    (latitude) float64 16B 49.73 51.13
  * longitude   (longitude) float64 24B 2.812 4.219 5.625
  * member      (member) int64 8B 0
Data variables:
    historical  (valid_time, latitude, longitude, member) float64 569kB 7.291...

In [34]:
#Load CMIP6 data
bncmip6_historical_all_2 = xr.open_dataset('/Users/anne-sophiesimons/Documents/Data thesis/CMIP6/CNRM-CM6-1/sfcWindmax_day_CNRM-CM6-1_historical_r2i1p1f2_gr_19500101-20141231.nc')
import xarray as xr

#Boundaries of Belgium
lat_min, lat_max = 49.5, 51.6
lon_min, lon_max = 2.5, 6.5

#Select only lon and lat of Belgium
ds_belgium_hist_all_2 = bncmip6_historical_all_2.sel(
    lat=slice(lat_min, lat_max),
    lon=slice(lon_min, lon_max)
)

bncmip6_hist_all_2 = ds_belgium_hist_all_2

winter_months = [1, 2, 3, 10, 11, 12]

months_hist_all_2 = ds_belgium_hist_all_2.sel(
    time=ds_belgium_hist_all_2.time.dt.month.isin(winter_months)
)

#Filter time between 2000-01-01 and 2014-12-31
#start_date = np.datetime64('1985-01-01')
#end_date   = np.datetime64('2015-01-01')

#ds_belgium_hist_all = bncmip6_hist_all.sel(
 #   time=slice(start_date, end_date)
#)


daysid_hist_all_2 = pd.to_datetime(months_hist_all_2['time'].values)

da_corrected_hist_all_2 = xr.DataArray(
    np.zeros([len(daysid_hist_all_2), latout.size, lonout.size, len(memid)]),
    coords=[daysid_hist_all_2, latout, lonout, memid],
    dims=['time', 'lat', 'lon', 'member'],
    name=pastname
)


df_corrected_hist_all_2 = da_corrected_hist_all_2.to_dataframe().reset_index()
df_corrected_hist_all_2

,time,lat,lon,member,historical
0,1950-01-01 12:00:00,49.727115,2.81250,0,0.0
1,1950-01-01 12:00:00,49.727115,4.21875,0,0.0
2,1950-01-01 12:00:00,49.727115,5.62500,0,0.0
3,1950-01-01 12:00:00,51.127867,2.81250,0,0.0
4,1950-01-01 12:00:00,51.127867,4.21875,0,0.0
...,...,...,...,...,...
71071,2014-12-31 12:00:00,49.727115,4.21875,0,0.0
71072,2014-12-31 12:00:00,49.727115,5.62500,0,0.0
71073,2014-12-31 12:00:00,51.127867,2.81250,0,0.0
71074,2014-12-31 12:00:00,51.127867,4.21875,0,0.0


In [35]:
# Start bias correctie
start_time = timer()

for ilat in latout:
    for ilon in lonout:
        #Selecteer GCM data op deze gridpoint
        ncdfw_point = bncmip6_historical.sel(lat=ilat, lon=ilon)
        ncdfw_df = bncmip6_historical.sel(lat=ilat, lon=ilon).to_dataframe().drop(['lat','lon'], axis=1).unstack()

        #Observaties van ERA5 op deze gridpoint
        dat_obs = era_qt_rg.sel(lat=ilat, lon=ilon, method = 'nearest').to_dataframe().drop(['lat','lon'], axis=1)['fg10']
        dat_mod = bncmip6_historical_2.sel(lat=ilat, lon=ilon, method = 'nearest').to_dataframe().drop(['lat','lon'], axis=1)['sfcWindmax'].unstack()
        dat_mod_all = months_hist_all_2.sel(lat=ilat, lon=ilon, method = 'nearest').to_dataframe().drop(['lat','lon'], axis=1)['sfcWindmax'].unstack()
        #Maar 1 kolom kiezen, want er zijn 2 identieke kolommen
        dat_mod = dat_mod.iloc[:, 0]
        dat_mod_all = dat_mod_all.iloc[:, 0]
        dat_mod.index = dat_mod.index.normalize()  # ERA5 en cmip6 moeten dezelfde tijden bevatten. Dit is ook zo maar bij cmip6 staat er nog 12:00 bij en dat moet men wegdoen
        dat_mod_all.index = dat_mod_all.index.normalize() 

        #dat_mod.index = pd.to_datetime(daysid.values)
        # Bias correctie met enkelvoudige tijdreeks

        # Reindex op ERA5 tijd
        #Alleen de overlappende periode overhouden tussen ERA5 en CMIP6 (2000-2004)
        dat_mod = dat_mod.reindex(dat_obs.index).dropna()

        # Bias correctie
        dat_mod_corrected_2 = bias_correct_time_series(dat_mod, dat_obs, dat_mod_all) 

        # Opslaan in DataArray
        ###Hier zit ergens een fout in ####    
        da_corrected_hist_all_2.loc[dict(lat=ilat, lon=ilon, member=0)] = dat_mod_corrected_2.values
        #da_corrected.loc[dict(lat=ilat, lon=ilon)] = dat_mod_corrected.values

        print(ilat)
        print(ilon)

end_time = timer()
print("Tijd voor bias correctie:", end_time - start_time)

<xarray.DataArray 'lat' ()> Size: 8B
array(49.72711456)
Coordinates:
    lat      float64 8B 49.73
    height   float64 8B 10.0
Attributes:
    axis:           Y
    standard_name:  latitude
    long_name:      Latitude
    units:          degrees_north
<xarray.DataArray 'lon' ()> Size: 8B
array(2.8125)
Coordinates:
    lon      float64 8B 2.812
    height   float64 8B 10.0
Attributes:
    axis:           X
    standard_name:  longitude
    long_name:      Longitude
    units:          degrees_east
<xarray.DataArray 'lat' ()> Size: 8B
array(49.72711456)
Coordinates:
    lat      float64 8B 49.73
    height   float64 8B 10.0
Attributes:
    axis:           Y
    standard_name:  latitude
    long_name:      Latitude
    units:          degrees_north
<xarray.DataArray 'lon' ()> Size: 8B
array(4.21875)
Coordinates:
    lon      float64 8B 4.219
    height   float64 8B 10.0
Attributes:
    axis:           X
    standard_name:  longitude
    long_name:      Longitude
    units:          degr

In [36]:
#De namen veranderen zodat de onderliggende functies gewoon kunnen blijven draaien
da_corrected_hist_all_2 = da_corrected_hist_all_2.rename({"lat": "latitude", "lon": "longitude"})
da_corrected_hist_all_2 = da_corrected_hist_all_2.rename({"time": "valid_time"})

df_corrected_historical_all_2 = da_corrected_hist_all_2.to_dataframe().reset_index()


bias_corr_ds_historical_all_2 = da_corrected_hist_all_2.to_dataset()

# Opslaan
bias_corr_ds_historical_all_2.to_netcdf('/Users/anne-sophiesimons/Documents/Data thesis/Bias Corrected data/België/bias_corr_historical_2')

In [37]:
bias_corr_historical_2 = xr.load_dataset('/Users/anne-sophiesimons/Documents/Data thesis/Bias Corrected data/België/bias_corr_historical_2')

In [38]:
bias_corr_historical_2

<xarray.Dataset> Size: 663kB
Dimensions:     (valid_time: 11846, latitude: 2, longitude: 3, member: 1)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 95kB 1950-01-01T12:00:00 ... 2014...
  * latitude    (latitude) float64 16B 49.73 51.13
  * longitude   (longitude) float64 24B 2.812 4.219 5.625
  * member      (member) int64 8B 0
Data variables:
    historical  (valid_time, latitude, longitude, member) float64 569kB 13.2 ...

In [41]:
#Load CMIP6 data
bncmip6_historical_3 = xr.open_dataset('/Users/anne-sophiesimons/Documents/Data thesis/CMIP6/CNRM-CM6-1/sfcWindmax_day_CNRM-CM6-1_historical_r3i1p1f2_gr_19500101-20141231.nc')
bncmip6_historical_3

#Boundaries of Belgium
lat_min, lat_max = 49.5, 51.6
lon_min, lon_max = 2.5, 6.5

#Select the lat and lon of Belgium
ds_belgium_historical_3 = bncmip6_historical_3.sel(
    lat=slice(lat_min, lat_max),
    lon=slice(lon_min, lon_max)
)

winter_months = [1, 2, 3, 10, 11, 12]

ds_belgium_historical_3 = ds_belgium_historical_3.sel(
    time=ds_belgium_historical_3.time.dt.month.isin(winter_months)
)

#Result
print(ds_belgium_historical_3)

#Set to dataframe
df = ds_belgium_historical_3.to_dataframe().reset_index()

import xarray as xr
import numpy as np
from timeit import default_timer as timer

#Open datasets
bncmip6_historical_3 = ds_belgium_historical_3
fnera5 = ds_daily_max

#Parameters
savencdf = True
pastname = "historical"  # naam van je historische data in bncmip6
modname = "GCM_model"    # geef hier je modelnaam op
data_folder = "/Users/axelleclaes/Desktop/wetransfer_thesis_2025-11-07_0852/bias_corrected"

#ERA5 data
era_da = fnera5['fg10']
era_df = era_da.to_dataframe().reset_index()

#lat en long van bncmip6 dataset nodig want we gaan daarop moeten correcten dus alles komt op deze grids terecht
latout = bncmip6_historical_3.lat
lonout = bncmip6_historical_3.lon
#Alle tijdstappenin het cmip6 bestand
daysid = bncmip6_historical_3.time
memid = bncmip6_historical_3.member if "member" in bncmip6_historical_3.dims else [0]  # check of er leden zijn
memid
daysid
latout
lonout

<xarray.Dataset> Size: 569kB
Dimensions:      (lat: 2, lon: 3, time: 11846, axis_nbounds: 2)
Coordinates:
  * lat          (lat) float64 16B 49.73 51.13
  * lon          (lon) float64 24B 2.812 4.219 5.625
    height       float64 8B ...
  * time         (time) datetime64[ns] 95kB 1950-01-01T12:00:00 ... 2014-12-3...
Dimensions without coordinates: axis_nbounds
Data variables:
    time_bounds  (time, axis_nbounds) datetime64[ns] 190kB ...
    sfcWindmax   (time, lat, lon) float32 284kB ...
Attributes: (12/53)
    Conventions:            CF-1.7 CMIP-6.2
    creation_date:          2018-07-14T10:45:59Z
    description:            CMIP6 historical
    title:                  CNRM-CM6-1 model output prepared for CMIP6 / CMIP...
    activity_id:            CMIP
    contact:                contact.cmip@meteo.fr
    ...                     ...
    nemo_gelato_commit:     49095b3accd5d4c_6524fe19b00467a
    arpege_minor_version:   6.3.2
    NCO:                    "4.5.5"
    parent_variant_la

<xarray.DataArray 'lon' (lon: 3)> Size: 24B
array([2.8125 , 4.21875, 5.625  ])
Coordinates:
  * lon      (lon) float64 24B 2.812 4.219 5.625
    height   float64 8B 10.0
Attributes:
    axis:           X
    standard_name:  longitude
    long_name:      Longitude
    units:          degrees_east

In [42]:
#Load CMIP6 data
bncmip6_historical_all_3 = xr.open_dataset('/Users/anne-sophiesimons/Documents/Data thesis/CMIP6/CNRM-CM6-1/sfcWindmax_day_CNRM-CM6-1_historical_r3i1p1f2_gr_19500101-20141231.nc')
import xarray as xr

#Boundaries of Belgium
lat_min, lat_max = 49.5, 51.6
lon_min, lon_max = 2.5, 6.5

#Select only lon and lat of Belgium
ds_belgium_hist_all_3 = bncmip6_historical_all_3.sel(
    lat=slice(lat_min, lat_max),
    lon=slice(lon_min, lon_max)
)

bncmip6_hist_all_3 = ds_belgium_hist_all_3

winter_months = [1, 2, 3, 10, 11, 12]

months_hist_all_3 = ds_belgium_hist_all_3.sel(
    time=ds_belgium_hist_all_3.time.dt.month.isin(winter_months)
)

daysid_hist_all_3 = pd.to_datetime(months_hist_all_3['time'].values)

da_corrected_hist_all_3 = xr.DataArray(
    np.zeros([len(daysid_hist_all_3), latout.size, lonout.size, len(memid)]),
    coords=[daysid_hist_all_3, latout, lonout, memid],
    dims=['time', 'lat', 'lon', 'member'],
    name=pastname
)

df_corrected_hist_all_3 = da_corrected_hist_all_3.to_dataframe().reset_index()
df_corrected_hist_all_3

,time,lat,lon,member,historical
0,1950-01-01 12:00:00,49.727115,2.81250,0,0.0
1,1950-01-01 12:00:00,49.727115,4.21875,0,0.0
2,1950-01-01 12:00:00,49.727115,5.62500,0,0.0
3,1950-01-01 12:00:00,51.127867,2.81250,0,0.0
4,1950-01-01 12:00:00,51.127867,4.21875,0,0.0
...,...,...,...,...,...
71071,2014-12-31 12:00:00,49.727115,4.21875,0,0.0
71072,2014-12-31 12:00:00,49.727115,5.62500,0,0.0
71073,2014-12-31 12:00:00,51.127867,2.81250,0,0.0
71074,2014-12-31 12:00:00,51.127867,4.21875,0,0.0


In [43]:
# Start bias correctie
start_time = timer()

for ilat in latout:
    for ilon in lonout:
        #Selecteer GCM data op deze gridpoint
        ncdfw_point = bncmip6_historical.sel(lat=ilat, lon=ilon)
        ncdfw_df = bncmip6_historical.sel(lat=ilat, lon=ilon).to_dataframe().drop(['lat','lon'], axis=1).unstack()

        #Observaties van ERA5 op deze gridpoint
        dat_obs = era_qt_rg.sel(lat=ilat, lon=ilon, method = 'nearest').to_dataframe().drop(['lat','lon'], axis=1)['fg10']
        dat_mod = bncmip6_historical_3.sel(lat=ilat, lon=ilon, method = 'nearest').to_dataframe().drop(['lat','lon'], axis=1)['sfcWindmax'].unstack()
        dat_mod_all = months_hist_all_3.sel(lat=ilat, lon=ilon, method = 'nearest').to_dataframe().drop(['lat','lon'], axis=1)['sfcWindmax'].unstack()
        #Maar 1 kolom kiezen, want er zijn 2 identieke kolommen
        dat_mod = dat_mod.iloc[:, 0]
        dat_mod_all = dat_mod_all.iloc[:, 0]
        dat_mod.index = dat_mod.index.normalize()  # ERA5 en cmip6 moeten dezelfde tijden bevatten. Dit is ook zo maar bij cmip6 staat er nog 12:00 bij en dat moet men wegdoen
        dat_mod_all.index = dat_mod_all.index.normalize() 

        #dat_mod.index = pd.to_datetime(daysid.values)
        # Bias correctie met enkelvoudige tijdreeks

        # Reindex op ERA5 tijd
        #Alleen de overlappende periode overhouden tussen ERA5 en CMIP6 (2000-2004)
        dat_mod = dat_mod.reindex(dat_obs.index).dropna()

        # Bias correctie
        dat_mod_corrected_3 = bias_correct_time_series(dat_mod, dat_obs, dat_mod_all) 

        # Opslaan in DataArray
        ###Hier zit ergens een fout in ####    
        da_corrected_hist_all_3.loc[dict(lat=ilat, lon=ilon, member=0)] = dat_mod_corrected_3.values
        #da_corrected.loc[dict(lat=ilat, lon=ilon)] = dat_mod_corrected.values

        print(ilat)
        print(ilon)

end_time = timer()
print("Tijd voor bias correctie:", end_time - start_time)

<xarray.DataArray 'lat' ()> Size: 8B
array(49.72711456)
Coordinates:
    lat      float64 8B 49.73
    height   float64 8B 10.0
Attributes:
    axis:           Y
    standard_name:  latitude
    long_name:      Latitude
    units:          degrees_north
<xarray.DataArray 'lon' ()> Size: 8B
array(2.8125)
Coordinates:
    lon      float64 8B 2.812
    height   float64 8B 10.0
Attributes:
    axis:           X
    standard_name:  longitude
    long_name:      Longitude
    units:          degrees_east
<xarray.DataArray 'lat' ()> Size: 8B
array(49.72711456)
Coordinates:
    lat      float64 8B 49.73
    height   float64 8B 10.0
Attributes:
    axis:           Y
    standard_name:  latitude
    long_name:      Latitude
    units:          degrees_north
<xarray.DataArray 'lon' ()> Size: 8B
array(4.21875)
Coordinates:
    lon      float64 8B 4.219
    height   float64 8B 10.0
Attributes:
    axis:           X
    standard_name:  longitude
    long_name:      Longitude
    units:          degr

In [44]:
#De namen veranderen zodat de onderliggende functies gewoon kunnen blijven draaien
da_corrected_hist_all_3 = da_corrected_hist_all_3.rename({"lat": "latitude", "lon": "longitude"})
da_corrected_hist_all_3 = da_corrected_hist_all_3.rename({"time": "valid_time"})

df_corrected_historical_all_3 = da_corrected_hist_all_3.to_dataframe().reset_index()


bias_corr_ds_historical_all_3 = da_corrected_hist_all_3.to_dataset()

# Opslaan
bias_corr_ds_historical_all_3.to_netcdf('/Users/anne-sophiesimons/Documents/Data thesis/Bias Corrected data/België/bias_corr_historical_3')

In [45]:
bias_corr_historical_3 = xr.load_dataset('/Users/anne-sophiesimons/Documents/Data thesis/Bias Corrected data/België/bias_corr_historical_3')

In [46]:
bias_corr_historical_3

<xarray.Dataset> Size: 663kB
Dimensions:     (valid_time: 11846, latitude: 2, longitude: 3, member: 1)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 95kB 1950-01-01T12:00:00 ... 2014...
  * latitude    (latitude) float64 16B 49.73 51.13
  * longitude   (longitude) float64 24B 2.812 4.219 5.625
  * member      (member) int64 8B 0
Data variables:
    historical  (valid_time, latitude, longitude, member) float64 569kB 11.83...

In [47]:
#Load CMIP6 data
bncmip6_historical_4 = xr.open_dataset('/Users/anne-sophiesimons/Documents/Data thesis/CMIP6/CNRM-CM6-1/sfcWindmax_day_CNRM-CM6-1_historical_r4i1p1f2_gr_19500101-20141231.nc')
bncmip6_historical_4

#Boundaries of Belgium
lat_min, lat_max = 49.5, 51.6
lon_min, lon_max = 2.5, 6.5

#Select the lat and lon of Belgium
ds_belgium_historical_4 = bncmip6_historical_4.sel(
    lat=slice(lat_min, lat_max),
    lon=slice(lon_min, lon_max)
)

winter_months = [1, 2, 3, 10, 11, 12]

ds_belgium_historical_4 = ds_belgium_historical_4.sel(
    time=ds_belgium_historical_4.time.dt.month.isin(winter_months)
)


#Result
print(ds_belgium_historical_4)

#Set to dataframe
df = ds_belgium_historical_4.to_dataframe().reset_index()

import xarray as xr
import numpy as np
from timeit import default_timer as timer

#Open datasets
bncmip6_historical_4 = ds_belgium_historical_4
fnera5 = ds_daily_max

#Parameters
savencdf = True
pastname = "historical"  # naam van je historische data in bncmip6
modname = "GCM_model"    # geef hier je modelnaam op
data_folder = "/Users/axelleclaes/Desktop/wetransfer_thesis_2025-11-07_0852/bias_corrected"

#ERA5 data
era_da = fnera5['fg10']
era_df = era_da.to_dataframe().reset_index()

#lat en long van bncmip6 dataset nodig want we gaan daarop moeten correcten dus alles komt op deze grids terecht
latout = bncmip6_historical_4.lat
lonout = bncmip6_historical_4.lon
#Alle tijdstappenin het cmip6 bestand
daysid = bncmip6_historical_4.time
memid = bncmip6_historical_4.member if "member" in bncmip6_historical_4.dims else [0]  # check of er leden zijn
memid
daysid
latout
lonout

<xarray.Dataset> Size: 569kB
Dimensions:      (lat: 2, lon: 3, time: 11846, axis_nbounds: 2)
Coordinates:
  * lat          (lat) float64 16B 49.73 51.13
  * lon          (lon) float64 24B 2.812 4.219 5.625
    height       float64 8B ...
  * time         (time) datetime64[ns] 95kB 1950-01-01T12:00:00 ... 2014-12-3...
Dimensions without coordinates: axis_nbounds
Data variables:
    time_bounds  (time, axis_nbounds) datetime64[ns] 190kB ...
    sfcWindmax   (time, lat, lon) float32 284kB ...
Attributes: (12/53)
    Conventions:            CF-1.7 CMIP-6.2
    creation_date:          2018-07-08T17:05:44Z
    description:            CMIP6 historical
    title:                  CNRM-CM6-1 model output prepared for CMIP6 / CMIP...
    activity_id:            CMIP
    contact:                contact.cmip@meteo.fr
    ...                     ...
    nemo_gelato_commit:     49095b3accd5d4c_6524fe19b00467a
    arpege_minor_version:   6.3.2
    NCO:                    "4.5.5"
    parent_variant_la

<xarray.DataArray 'lon' (lon: 3)> Size: 24B
array([2.8125 , 4.21875, 5.625  ])
Coordinates:
  * lon      (lon) float64 24B 2.812 4.219 5.625
    height   float64 8B 10.0
Attributes:
    axis:           X
    standard_name:  longitude
    long_name:      Longitude
    units:          degrees_east

In [48]:
#Load CMIP6 data
bncmip6_historical_all_4 = xr.open_dataset('/Users/anne-sophiesimons/Documents/Data thesis/CMIP6/CNRM-CM6-1/sfcWindmax_day_CNRM-CM6-1_historical_r4i1p1f2_gr_19500101-20141231.nc')
import xarray as xr

#Boundaries of Belgium
lat_min, lat_max = 49.5, 51.6
lon_min, lon_max = 2.5, 6.5

#Select only lon and lat of Belgium
ds_belgium_hist_all_4 = bncmip6_historical_all_4.sel(
    lat=slice(lat_min, lat_max),
    lon=slice(lon_min, lon_max)
)

bncmip6_hist_all_4 = ds_belgium_hist_all_4

winter_months = [1, 2, 3, 10, 11, 12]

months_hist_all_4 = ds_belgium_hist_all_4.sel(
    time=ds_belgium_hist_all_4.time.dt.month.isin(winter_months)
)

daysid_hist_all_4 = pd.to_datetime(months_hist_all_4['time'].values)

da_corrected_hist_all_4 = xr.DataArray(
    np.zeros([len(daysid_hist_all_4), latout.size, lonout.size, len(memid)]),
    coords=[daysid_hist_all_3, latout, lonout, memid],
    dims=['time', 'lat', 'lon', 'member'],
    name=pastname
)

df_corrected_hist_all_4 = da_corrected_hist_all_4.to_dataframe().reset_index()
df_corrected_hist_all_4

,time,lat,lon,member,historical
0,1950-01-01 12:00:00,49.727115,2.81250,0,0.0
1,1950-01-01 12:00:00,49.727115,4.21875,0,0.0
2,1950-01-01 12:00:00,49.727115,5.62500,0,0.0
3,1950-01-01 12:00:00,51.127867,2.81250,0,0.0
4,1950-01-01 12:00:00,51.127867,4.21875,0,0.0
...,...,...,...,...,...
71071,2014-12-31 12:00:00,49.727115,4.21875,0,0.0
71072,2014-12-31 12:00:00,49.727115,5.62500,0,0.0
71073,2014-12-31 12:00:00,51.127867,2.81250,0,0.0
71074,2014-12-31 12:00:00,51.127867,4.21875,0,0.0


In [49]:
# Start bias correctie
start_time = timer()

for ilat in latout:
    for ilon in lonout:
        #Selecteer GCM data op deze gridpoint
        ncdfw_point = bncmip6_historical.sel(lat=ilat, lon=ilon)
        ncdfw_df = bncmip6_historical.sel(lat=ilat, lon=ilon).to_dataframe().drop(['lat','lon'], axis=1).unstack()

        #Observaties van ERA5 op deze gridpoint
        dat_obs = era_qt_rg.sel(lat=ilat, lon=ilon, method = 'nearest').to_dataframe().drop(['lat','lon'], axis=1)['fg10']
        dat_mod = bncmip6_historical_4.sel(lat=ilat, lon=ilon, method = 'nearest').to_dataframe().drop(['lat','lon'], axis=1)['sfcWindmax'].unstack()
        dat_mod_all = months_hist_all_4.sel(lat=ilat, lon=ilon, method = 'nearest').to_dataframe().drop(['lat','lon'], axis=1)['sfcWindmax'].unstack()
        #Maar 1 kolom kiezen, want er zijn 2 identieke kolommen
        dat_mod = dat_mod.iloc[:, 0]
        dat_mod_all = dat_mod_all.iloc[:, 0]
        dat_mod.index = dat_mod.index.normalize()  # ERA5 en cmip6 moeten dezelfde tijden bevatten. Dit is ook zo maar bij cmip6 staat er nog 12:00 bij en dat moet men wegdoen
        dat_mod_all.index = dat_mod_all.index.normalize() 

        #dat_mod.index = pd.to_datetime(daysid.values)
        # Bias correctie met enkelvoudige tijdreeks

        # Reindex op ERA5 tijd
        #Alleen de overlappende periode overhouden tussen ERA5 en CMIP6 (2000-2004)
        dat_mod = dat_mod.reindex(dat_obs.index).dropna()

        # Bias correctie
        dat_mod_corrected_4 = bias_correct_time_series(dat_mod, dat_obs, dat_mod_all) 

        # Opslaan in DataArray
        ###Hier zit ergens een fout in ####    
        da_corrected_hist_all_4.loc[dict(lat=ilat, lon=ilon, member=0)] = dat_mod_corrected_4.values
        #da_corrected.loc[dict(lat=ilat, lon=ilon)] = dat_mod_corrected.values

        print(ilat)
        print(ilon)

end_time = timer()
print("Tijd voor bias correctie:", end_time - start_time)

<xarray.DataArray 'lat' ()> Size: 8B
array(49.72711456)
Coordinates:
    lat      float64 8B 49.73
    height   float64 8B 10.0
Attributes:
    axis:           Y
    standard_name:  latitude
    long_name:      Latitude
    units:          degrees_north
<xarray.DataArray 'lon' ()> Size: 8B
array(2.8125)
Coordinates:
    lon      float64 8B 2.812
    height   float64 8B 10.0
Attributes:
    axis:           X
    standard_name:  longitude
    long_name:      Longitude
    units:          degrees_east
<xarray.DataArray 'lat' ()> Size: 8B
array(49.72711456)
Coordinates:
    lat      float64 8B 49.73
    height   float64 8B 10.0
Attributes:
    axis:           Y
    standard_name:  latitude
    long_name:      Latitude
    units:          degrees_north
<xarray.DataArray 'lon' ()> Size: 8B
array(4.21875)
Coordinates:
    lon      float64 8B 4.219
    height   float64 8B 10.0
Attributes:
    axis:           X
    standard_name:  longitude
    long_name:      Longitude
    units:          degr

In [50]:
#De namen veranderen zodat de onderliggende functies gewoon kunnen blijven draaien
da_corrected_hist_all_4 = da_corrected_hist_all_4.rename({"lat": "latitude", "lon": "longitude"})
da_corrected_hist_all_4 = da_corrected_hist_all_4.rename({"time": "valid_time"})

df_corrected_historical_all_4 = da_corrected_hist_all_4.to_dataframe().reset_index()


bias_corr_ds_historical_all_4 = da_corrected_hist_all_4.to_dataset()

# Opslaan
bias_corr_ds_historical_all_4.to_netcdf('/Users/anne-sophiesimons/Documents/Data thesis/Bias Corrected data/België/bias_corr_historical_4')

In [51]:
bias_corr_historical_4 = xr.load_dataset('/Users/anne-sophiesimons/Documents/Data thesis/Bias Corrected data/België/bias_corr_historical_4')

In [52]:
bias_corr_historical_4

<xarray.Dataset> Size: 663kB
Dimensions:     (valid_time: 11846, latitude: 2, longitude: 3, member: 1)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 95kB 1950-01-01T12:00:00 ... 2014...
  * latitude    (latitude) float64 16B 49.73 51.13
  * longitude   (longitude) float64 24B 2.812 4.219 5.625
  * member      (member) int64 8B 0
Data variables:
    historical  (valid_time, latitude, longitude, member) float64 569kB 14.26...

In [53]:
#Load CMIP6 data under the scenario SSP 585
bncmip6_ssp585 = xr.open_dataset('/Users/anne-sophiesimons/Documents/Data thesis/CMIP6/CNRM-CM6-1/sfcWindmax_day_CNRM-CM6-1_ssp585_r1i1p1f2_gr_20150101-21001231.nc')
winter_months = [1, 2, 3, 10, 11, 12]
bncmip6_ssp585_winter = bncmip6_ssp585.sel(
    time=bncmip6_ssp585.time.dt.month.isin(winter_months)
)

#Boundaries of Belgium
lat_min, lat_max = 49.5, 51.6
lon_min, lon_max = 2.5, 6.5

#Select only lon and lat of Belgium
ds_belgium_ssp585 = bncmip6_ssp585_winter.sel(
    lat=slice(lat_min, lat_max),
    lon=slice(lon_min, lon_max)
)

bncmip6_ssp585 = ds_belgium_ssp585

winter_months = [1, 2, 3, 10, 11, 12]

months_ssp585 = ds_belgium_ssp585.sel(
    time=ds_belgium_ssp585.time.dt.month.isin(winter_months)
)

daysid_ssp585 = pd.to_datetime(months_ssp585['time'].values)

da_corrected_ssp585 = xr.DataArray(
    np.zeros([len(daysid_ssp585), latout.size, lonout.size, len(memid)]),
    coords=[daysid_ssp585, latout, lonout, memid],
    dims=['time', 'lat', 'lon', 'member'],
    name=pastname
)


df_corrected_ssp585 = da_corrected_ssp585.to_dataframe().reset_index()
df_corrected_ssp585

,time,lat,lon,member,historical
0,2015-01-01 12:00:00,49.727115,2.81250,0,0.0
1,2015-01-01 12:00:00,49.727115,4.21875,0,0.0
2,2015-01-01 12:00:00,49.727115,5.62500,0,0.0
3,2015-01-01 12:00:00,51.127867,2.81250,0,0.0
4,2015-01-01 12:00:00,51.127867,4.21875,0,0.0
...,...,...,...,...,...
94033,2100-12-31 12:00:00,49.727115,4.21875,0,0.0
94034,2100-12-31 12:00:00,49.727115,5.62500,0,0.0
94035,2100-12-31 12:00:00,51.127867,2.81250,0,0.0
94036,2100-12-31 12:00:00,51.127867,4.21875,0,0.0


In [54]:
# Start bias correctie
start_time = timer()

for ilat in latout:
    for ilon in lonout:
        #Selecteer GCM data op deze gridpoint
        ncdfw_point = bncmip6_historical.sel(lat=ilat, lon=ilon)
        ncdfw_df = bncmip6_historical.sel(lat=ilat, lon=ilon).to_dataframe().drop(['lat','lon'], axis=1).unstack()

        #Observaties van ERA5 op deze gridpoint
        dat_obs = era_qt_rg.sel(lat=ilat, lon=ilon, method = 'nearest').to_dataframe().drop(['lat','lon'], axis=1)['fg10']
        dat_mod = bncmip6_historical.sel(lat=ilat, lon=ilon, method = 'nearest').to_dataframe().drop(['lat','lon'], axis=1)['sfcWindmax'].unstack()
        dat_mod_ssp585 = bncmip6_ssp585_winter.sel(lat=ilat, lon=ilon, method = 'nearest').to_dataframe().drop(['lat','lon'], axis=1)['sfcWindmax'].unstack()
        #Maar 1 kolom kiezen, want er zijn 2 identieke kolommen
        dat_mod = dat_mod.iloc[:, 0]
        dat_mod_ssp585 = dat_mod_ssp585.iloc[:, 0]
        dat_mod.index = dat_mod.index.normalize()  # ERA5 en cmip6 moeten dezelfde tijden bevatten. Dit is ook zo maar bij cmip6 staat er nog 12:00 bij en dat moet men wegdoen
        dat_mod_ssp585.index = dat_mod_ssp585.index.normalize() 

        #dat_mod.index = pd.to_datetime(daysid.values)
        # Bias correctie met enkelvoudige tijdreeks

        # Reindex op ERA5 tijd
        #Alleen de overlappende periode overhouden tussen ERA5 en CMIP6 (2000-2004)
        dat_mod = dat_mod.reindex(dat_obs.index).dropna()

        # Bias correctie
        dat_mod_corrected = bias_correct_time_series(dat_mod, dat_obs, dat_mod_ssp585) 

        # Opslaan in DataArray
        ###Hier zit ergens een fout in ####    
        da_corrected_ssp585.loc[dict(lat=ilat, lon=ilon, member=0)] = dat_mod_corrected.values
        #da_corrected.loc[dict(lat=ilat, lon=ilon)] = dat_mod_corrected.values

        print(ilat)
        print(ilon)

end_time = timer()
print("Tijd voor bias correctie:", end_time - start_time)

<xarray.DataArray 'lat' ()> Size: 8B
array(49.72711456)
Coordinates:
    lat      float64 8B 49.73
    height   float64 8B 10.0
Attributes:
    axis:           Y
    standard_name:  latitude
    long_name:      Latitude
    units:          degrees_north
<xarray.DataArray 'lon' ()> Size: 8B
array(2.8125)
Coordinates:
    lon      float64 8B 2.812
    height   float64 8B 10.0
Attributes:
    axis:           X
    standard_name:  longitude
    long_name:      Longitude
    units:          degrees_east
<xarray.DataArray 'lat' ()> Size: 8B
array(49.72711456)
Coordinates:
    lat      float64 8B 49.73
    height   float64 8B 10.0
Attributes:
    axis:           Y
    standard_name:  latitude
    long_name:      Latitude
    units:          degrees_north
<xarray.DataArray 'lon' ()> Size: 8B
array(4.21875)
Coordinates:
    lon      float64 8B 4.219
    height   float64 8B 10.0
Attributes:
    axis:           X
    standard_name:  longitude
    long_name:      Longitude
    units:          degr

In [57]:
#De namen veranderen zodat de onderliggende functies gewoon kunnen blijven draaien
#da_corrected_ssp585 = da_corrected_ssp585.rename({"lat": "latitude", "lon": "longitude"})
#da_corrected_ssp585 = da_corrected_ssp585.rename({"time": "valid_time"})

df_corrected_ssp585 = da_corrected_ssp585.to_dataframe().reset_index()
df_corrected_ssp585

#df_corrected_ssp585.max() #Maximale waarde bias corrected CMIP6
bias_corr_ds_ssp585 = da_corrected_ssp585.to_dataset()

# Opslaan
bias_corr_ds_ssp585.to_netcdf('/Users/anne-sophiesimons/Documents/Data thesis/Bias Corrected data/België/bias_corr_ssp585_1')

In [58]:
bias_corr_ssp585_1 = xr.load_dataset('/Users/anne-sophiesimons/Documents/Data thesis/Bias Corrected data/België/bias_corr_ssp585_1')

In [59]:
bias_corr_ssp585_1

<xarray.Dataset> Size: 878kB
Dimensions:     (valid_time: 15673, latitude: 2, longitude: 3, member: 1)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 125kB 2015-01-01T12:00:00 ... 210...
  * latitude    (latitude) float64 16B 49.73 51.13
  * longitude   (longitude) float64 24B 2.812 4.219 5.625
  * member      (member) int64 8B 0
Data variables:
    historical  (valid_time, latitude, longitude, member) float64 752kB 5.163...

In [61]:
#Load CMIP6 data under the scenario SSP 585
bncmip6_ssp585_2 = xr.open_dataset('/Users/anne-sophiesimons/Documents/Data thesis/CMIP6/CNRM-CM6-1/sfcWindmax_day_CNRM-CM6-1_ssp585_r2i1p1f2_gr_20150101-21001231.nc')
winter_months = [1, 2, 3, 10, 11, 12]
bncmip6_ssp585_winter_2 = bncmip6_ssp585_2.sel(
    time=bncmip6_ssp585_2.time.dt.month.isin(winter_months)
)

#Boundaries of Belgium
lat_min, lat_max = 49.5, 51.6
lon_min, lon_max = 2.5, 6.5

#Select only lon and lat of Belgium
ds_belgium_ssp585_2 = bncmip6_ssp585_winter_2.sel(
    lat=slice(lat_min, lat_max),
    lon=slice(lon_min, lon_max)
)

bncmip6_ssp585_2 = ds_belgium_ssp585_2

winter_months = [1, 2, 3, 10, 11, 12]

months_ssp585_2 = ds_belgium_ssp585_2.sel(
    time=ds_belgium_ssp585_2.time.dt.month.isin(winter_months)
)

daysid_ssp585_2 = pd.to_datetime(months_ssp585_2['time'].values)

da_corrected_ssp585_2 = xr.DataArray(
    np.zeros([len(daysid_ssp585_2), latout.size, lonout.size, len(memid)]),
    coords=[daysid_ssp585_2, latout, lonout, memid],
    dims=['time', 'lat', 'lon', 'member'],
    name=pastname
)


df_corrected_ssp585_2 = da_corrected_ssp585_2.to_dataframe().reset_index()
df_corrected_ssp585_2

,time,lat,lon,member,historical
0,2015-01-01 12:00:00,49.727115,2.81250,0,0.0
1,2015-01-01 12:00:00,49.727115,4.21875,0,0.0
2,2015-01-01 12:00:00,49.727115,5.62500,0,0.0
3,2015-01-01 12:00:00,51.127867,2.81250,0,0.0
4,2015-01-01 12:00:00,51.127867,4.21875,0,0.0
...,...,...,...,...,...
94033,2100-12-31 12:00:00,49.727115,4.21875,0,0.0
94034,2100-12-31 12:00:00,49.727115,5.62500,0,0.0
94035,2100-12-31 12:00:00,51.127867,2.81250,0,0.0
94036,2100-12-31 12:00:00,51.127867,4.21875,0,0.0


In [62]:
# Start bias correctie
start_time = timer()

for ilat in latout:
    for ilon in lonout:
        #Selecteer GCM data op deze gridpoint
        ncdfw_point = bncmip6_historical.sel(lat=ilat, lon=ilon)
        ncdfw_df = bncmip6_historical.sel(lat=ilat, lon=ilon).to_dataframe().drop(['lat','lon'], axis=1).unstack()

        #Observaties van ERA5 op deze gridpoint
        dat_obs = era_qt_rg.sel(lat=ilat, lon=ilon, method = 'nearest').to_dataframe().drop(['lat','lon'], axis=1)['fg10']
        dat_mod = bncmip6_historical_2.sel(lat=ilat, lon=ilon, method = 'nearest').to_dataframe().drop(['lat','lon'], axis=1)['sfcWindmax'].unstack()
        dat_mod_ssp585_2 = bncmip6_ssp585_winter_2.sel(lat=ilat, lon=ilon, method = 'nearest').to_dataframe().drop(['lat','lon'], axis=1)['sfcWindmax'].unstack()
        #Maar 1 kolom kiezen, want er zijn 2 identieke kolommen
        dat_mod = dat_mod.iloc[:, 0]
        dat_mod_ssp585_2 = dat_mod_ssp585_2.iloc[:, 0]
        dat_mod.index = dat_mod.index.normalize()  # ERA5 en cmip6 moeten dezelfde tijden bevatten. Dit is ook zo maar bij cmip6 staat er nog 12:00 bij en dat moet men wegdoen
        dat_mod_ssp585_2.index = dat_mod_ssp585_2.index.normalize() 

        #dat_mod.index = pd.to_datetime(daysid.values)
        # Bias correctie met enkelvoudige tijdreeks

        # Reindex op ERA5 tijd
        #Alleen de overlappende periode overhouden tussen ERA5 en CMIP6 (2000-2004)
        dat_mod = dat_mod.reindex(dat_obs.index).dropna()

        # Bias correctie
        dat_mod_corrected_2 = bias_correct_time_series(dat_mod, dat_obs, dat_mod_ssp585_2) 

        # Opslaan in DataArray
        ###Hier zit ergens een fout in ####    
        da_corrected_ssp585_2.loc[dict(lat=ilat, lon=ilon, member=0)] = dat_mod_corrected_2.values
        #da_corrected.loc[dict(lat=ilat, lon=ilon)] = dat_mod_corrected.values

        print(ilat)
        print(ilon)

end_time = timer()
print("Tijd voor bias correctie:", end_time - start_time)

<xarray.DataArray 'lat' ()> Size: 8B
array(49.72711456)
Coordinates:
    lat      float64 8B 49.73
    height   float64 8B 10.0
Attributes:
    axis:           Y
    standard_name:  latitude
    long_name:      Latitude
    units:          degrees_north
<xarray.DataArray 'lon' ()> Size: 8B
array(2.8125)
Coordinates:
    lon      float64 8B 2.812
    height   float64 8B 10.0
Attributes:
    axis:           X
    standard_name:  longitude
    long_name:      Longitude
    units:          degrees_east
<xarray.DataArray 'lat' ()> Size: 8B
array(49.72711456)
Coordinates:
    lat      float64 8B 49.73
    height   float64 8B 10.0
Attributes:
    axis:           Y
    standard_name:  latitude
    long_name:      Latitude
    units:          degrees_north
<xarray.DataArray 'lon' ()> Size: 8B
array(4.21875)
Coordinates:
    lon      float64 8B 4.219
    height   float64 8B 10.0
Attributes:
    axis:           X
    standard_name:  longitude
    long_name:      Longitude
    units:          degr

In [63]:
#De namen veranderen zodat de onderliggende functies gewoon kunnen blijven draaien
da_corrected_ssp585_2 = da_corrected_ssp585_2.rename({"lat": "latitude", "lon": "longitude"})
da_corrected_ssp585_2 = da_corrected_ssp585_2.rename({"time": "valid_time"})

df_corrected_ssp585_2 = da_corrected_ssp585_2.to_dataframe().reset_index()
df_corrected_ssp585_2

#df_corrected_ssp585.max() #Maximale waarde bias corrected CMIP6
bias_corr_ds_ssp585_2 = da_corrected_ssp585_2.to_dataset()

# Opslaan
bias_corr_ds_ssp585_2.to_netcdf('/Users/anne-sophiesimons/Documents/Data thesis/Bias Corrected data/België/bias_corr_ssp585_2')

In [64]:
bias_corr_ssp585_2 = xr.load_dataset('/Users/anne-sophiesimons/Documents/Data thesis/Bias Corrected data/België/bias_corr_ssp585_2')

In [65]:
bias_corr_ssp585_2

<xarray.Dataset> Size: 878kB
Dimensions:     (valid_time: 15673, latitude: 2, longitude: 3, member: 1)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 125kB 2015-01-01T12:00:00 ... 210...
  * latitude    (latitude) float64 16B 49.73 51.13
  * longitude   (longitude) float64 24B 2.812 4.219 5.625
  * member      (member) int64 8B 0
Data variables:
    historical  (valid_time, latitude, longitude, member) float64 752kB 11.29...

In [66]:
#Load CMIP6 data under the scenario SSP 585
bncmip6_ssp585_3 = xr.open_dataset('/Users/anne-sophiesimons/Documents/Data thesis/CMIP6/CNRM-CM6-1/sfcWindmax_day_CNRM-CM6-1_ssp585_r3i1p1f2_gr_20150101-21001231.nc')
winter_months = [1, 2, 3, 10, 11, 12]
bncmip6_ssp585_winter_3 = bncmip6_ssp585_3.sel(
    time=bncmip6_ssp585_3.time.dt.month.isin(winter_months)
)

#Boundaries of Belgium
lat_min, lat_max = 49.5, 51.6
lon_min, lon_max = 2.5, 6.5

#Select only lon and lat of Belgium
ds_belgium_ssp585_3 = bncmip6_ssp585_winter_3.sel(
    lat=slice(lat_min, lat_max),
    lon=slice(lon_min, lon_max)
)

bncmip6_ssp585_3 = ds_belgium_ssp585_3

winter_months = [1, 2, 3, 10, 11, 12]

months_ssp585_3 = ds_belgium_ssp585_3.sel(
    time=ds_belgium_ssp585_3.time.dt.month.isin(winter_months)
)

daysid_ssp585_3 = pd.to_datetime(months_ssp585_3['time'].values)

da_corrected_ssp585_3 = xr.DataArray(
    np.zeros([len(daysid_ssp585_3), latout.size, lonout.size, len(memid)]),
    coords=[daysid_ssp585_3, latout, lonout, memid],
    dims=['time', 'lat', 'lon', 'member'],
    name=pastname
)


df_corrected_ssp585_3 = da_corrected_ssp585_3.to_dataframe().reset_index()
df_corrected_ssp585_3

,time,lat,lon,member,historical
0,2015-01-01 12:00:00,49.727115,2.81250,0,0.0
1,2015-01-01 12:00:00,49.727115,4.21875,0,0.0
2,2015-01-01 12:00:00,49.727115,5.62500,0,0.0
3,2015-01-01 12:00:00,51.127867,2.81250,0,0.0
4,2015-01-01 12:00:00,51.127867,4.21875,0,0.0
...,...,...,...,...,...
94033,2100-12-31 12:00:00,49.727115,4.21875,0,0.0
94034,2100-12-31 12:00:00,49.727115,5.62500,0,0.0
94035,2100-12-31 12:00:00,51.127867,2.81250,0,0.0
94036,2100-12-31 12:00:00,51.127867,4.21875,0,0.0


In [67]:
# Start bias correctie
start_time = timer()

for ilat in latout:
    for ilon in lonout:
        #Selecteer GCM data op deze gridpoint
        ncdfw_point = bncmip6_historical.sel(lat=ilat, lon=ilon)
        ncdfw_df = bncmip6_historical.sel(lat=ilat, lon=ilon).to_dataframe().drop(['lat','lon'], axis=1).unstack()

        #Observaties van ERA5 op deze gridpoint
        dat_obs = era_qt_rg.sel(lat=ilat, lon=ilon, method = 'nearest').to_dataframe().drop(['lat','lon'], axis=1)['fg10']
        dat_mod = bncmip6_historical_3.sel(lat=ilat, lon=ilon, method = 'nearest').to_dataframe().drop(['lat','lon'], axis=1)['sfcWindmax'].unstack()
        dat_mod_ssp585_3 = bncmip6_ssp585_winter_3.sel(lat=ilat, lon=ilon, method = 'nearest').to_dataframe().drop(['lat','lon'], axis=1)['sfcWindmax'].unstack()
        #Maar 1 kolom kiezen, want er zijn 2 identieke kolommen
        dat_mod = dat_mod.iloc[:, 0]
        dat_mod_ssp585_3 = dat_mod_ssp585_3.iloc[:, 0]
        dat_mod.index = dat_mod.index.normalize()  # ERA5 en cmip6 moeten dezelfde tijden bevatten. Dit is ook zo maar bij cmip6 staat er nog 12:00 bij en dat moet men wegdoen
        dat_mod_ssp585_3.index = dat_mod_ssp585_3.index.normalize() 

        #dat_mod.index = pd.to_datetime(daysid.values)
        # Bias correctie met enkelvoudige tijdreeks

        # Reindex op ERA5 tijd
        #Alleen de overlappende periode overhouden tussen ERA5 en CMIP6 (2000-2004)
        dat_mod = dat_mod.reindex(dat_obs.index).dropna()

        # Bias correctie
        dat_mod_corrected_3 = bias_correct_time_series(dat_mod, dat_obs, dat_mod_ssp585_3) 

        # Opslaan in DataArray
        ###Hier zit ergens een fout in ####    
        da_corrected_ssp585_3.loc[dict(lat=ilat, lon=ilon, member=0)] = dat_mod_corrected_3.values
        #da_corrected.loc[dict(lat=ilat, lon=ilon)] = dat_mod_corrected.values

        print(ilat)
        print(ilon)

end_time = timer()
print("Tijd voor bias correctie:", end_time - start_time)

<xarray.DataArray 'lat' ()> Size: 8B
array(49.72711456)
Coordinates:
    lat      float64 8B 49.73
    height   float64 8B 10.0
Attributes:
    axis:           Y
    standard_name:  latitude
    long_name:      Latitude
    units:          degrees_north
<xarray.DataArray 'lon' ()> Size: 8B
array(2.8125)
Coordinates:
    lon      float64 8B 2.812
    height   float64 8B 10.0
Attributes:
    axis:           X
    standard_name:  longitude
    long_name:      Longitude
    units:          degrees_east
<xarray.DataArray 'lat' ()> Size: 8B
array(49.72711456)
Coordinates:
    lat      float64 8B 49.73
    height   float64 8B 10.0
Attributes:
    axis:           Y
    standard_name:  latitude
    long_name:      Latitude
    units:          degrees_north
<xarray.DataArray 'lon' ()> Size: 8B
array(4.21875)
Coordinates:
    lon      float64 8B 4.219
    height   float64 8B 10.0
Attributes:
    axis:           X
    standard_name:  longitude
    long_name:      Longitude
    units:          degr

In [68]:
#De namen veranderen zodat de onderliggende functies gewoon kunnen blijven draaien
da_corrected_ssp585_3 = da_corrected_ssp585_3.rename({"lat": "latitude", "lon": "longitude"})
da_corrected_ssp585_3 = da_corrected_ssp585_3.rename({"time": "valid_time"})

df_corrected_ssp585_3 = da_corrected_ssp585_3.to_dataframe().reset_index()
df_corrected_ssp585_3

#df_corrected_ssp585.max() #Maximale waarde bias corrected CMIP6
bias_corr_ds_ssp585_3 = da_corrected_ssp585_3.to_dataset()

# Opslaan
bias_corr_ds_ssp585_3.to_netcdf('/Users/anne-sophiesimons/Documents/Data thesis/Bias Corrected data/België/bias_corr_ssp585_3')

In [69]:
bias_corr_ssp585_3 = xr.load_dataset('/Users/anne-sophiesimons/Documents/Data thesis/Bias Corrected data/België/bias_corr_ssp585_3')

In [70]:
bias_corr_ssp585_3

<xarray.Dataset> Size: 878kB
Dimensions:     (valid_time: 15673, latitude: 2, longitude: 3, member: 1)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 125kB 2015-01-01T12:00:00 ... 210...
  * latitude    (latitude) float64 16B 49.73 51.13
  * longitude   (longitude) float64 24B 2.812 4.219 5.625
  * member      (member) int64 8B 0
Data variables:
    historical  (valid_time, latitude, longitude, member) float64 752kB 15.98...

In [71]:
#Load CMIP6 data under the scenario SSP 585
bncmip6_ssp585_4 = xr.open_dataset('/Users/anne-sophiesimons/Documents/Data thesis/CMIP6/CNRM-CM6-1/sfcWindmax_day_CNRM-CM6-1_ssp585_r3i1p1f2_gr_20150101-21001231.nc')
winter_months = [1, 2, 3, 10, 11, 12]
bncmip6_ssp585_winter_4 = bncmip6_ssp585_4.sel(
    time=bncmip6_ssp585_4.time.dt.month.isin(winter_months)
)

#Boundaries of Belgium
lat_min, lat_max = 49.5, 51.6
lon_min, lon_max = 2.5, 6.5

#Select only lon and lat of Belgium
ds_belgium_ssp585_4 = bncmip6_ssp585_winter_4.sel(
    lat=slice(lat_min, lat_max),
    lon=slice(lon_min, lon_max)
)

bncmip6_ssp585_4 = ds_belgium_ssp585_4

winter_months = [1, 2, 3, 10, 11, 12]

months_ssp585_4 = ds_belgium_ssp585_4.sel(
    time=ds_belgium_ssp585_4.time.dt.month.isin(winter_months)
)

daysid_ssp585_4 = pd.to_datetime(months_ssp585_4['time'].values)

da_corrected_ssp585_4 = xr.DataArray(
    np.zeros([len(daysid_ssp585_4), latout.size, lonout.size, len(memid)]),
    coords=[daysid_ssp585_4, latout, lonout, memid],
    dims=['time', 'lat', 'lon', 'member'],
    name=pastname
)


df_corrected_ssp585_4 = da_corrected_ssp585_4.to_dataframe().reset_index()
df_corrected_ssp585_4

,time,lat,lon,member,historical
0,2015-01-01 12:00:00,49.727115,2.81250,0,0.0
1,2015-01-01 12:00:00,49.727115,4.21875,0,0.0
2,2015-01-01 12:00:00,49.727115,5.62500,0,0.0
3,2015-01-01 12:00:00,51.127867,2.81250,0,0.0
4,2015-01-01 12:00:00,51.127867,4.21875,0,0.0
...,...,...,...,...,...
94033,2100-12-31 12:00:00,49.727115,4.21875,0,0.0
94034,2100-12-31 12:00:00,49.727115,5.62500,0,0.0
94035,2100-12-31 12:00:00,51.127867,2.81250,0,0.0
94036,2100-12-31 12:00:00,51.127867,4.21875,0,0.0


In [75]:
# Start bias correctie
start_time = timer()

for ilat in latout:
    for ilon in lonout:
        #Selecteer GCM data op deze gridpoint
        ncdfw_point = bncmip6_historical.sel(lat=ilat, lon=ilon)
        ncdfw_df = bncmip6_historical.sel(lat=ilat, lon=ilon).to_dataframe().drop(['lat','lon'], axis=1).unstack()

        #Observaties van ERA5 op deze gridpoint
        dat_obs = era_qt_rg.sel(lat=ilat, lon=ilon, method = 'nearest').to_dataframe().drop(['lat','lon'], axis=1)['fg10']
        dat_mod = bncmip6_historical_4.sel(lat=ilat, lon=ilon, method = 'nearest').to_dataframe().drop(['lat','lon'], axis=1)['sfcWindmax'].unstack()
        dat_mod_ssp585_4 = bncmip6_ssp585_winter_4.sel(lat=ilat, lon=ilon, method = 'nearest').to_dataframe().drop(['lat','lon'], axis=1)['sfcWindmax'].unstack()
        #Maar 1 kolom kiezen, want er zijn 2 identieke kolommen
        dat_mod = dat_mod.iloc[:, 0]
        dat_mod_ssp585_4 = dat_mod_ssp585_4.iloc[:, 0]
        dat_mod.index = dat_mod.index.normalize()  # ERA5 en cmip6 moeten dezelfde tijden bevatten. Dit is ook zo maar bij cmip6 staat er nog 12:00 bij en dat moet men wegdoen
        dat_mod_ssp585_4.index = dat_mod_ssp585_4.index.normalize() 

        #dat_mod.index = pd.to_datetime(daysid.values)
        # Bias correctie met enkelvoudige tijdreeks

        # Reindex op ERA5 tijd
        #Alleen de overlappende periode overhouden tussen ERA5 en CMIP6 (2000-2004)
        dat_mod = dat_mod.reindex(dat_obs.index).dropna()

        # Bias correctie
        dat_mod_corrected_4 = bias_correct_time_series(dat_mod, dat_obs, dat_mod_ssp585_4) 

        # Opslaan in DataArray
        ###Hier zit ergens een fout in ####    
        da_corrected_ssp585_4.loc[dict(lat=ilat, lon=ilon, member=0)] = dat_mod_corrected_4.values
        #da_corrected.loc[dict(lat=ilat, lon=ilon)] = dat_mod_corrected.values

        print(ilat)
        print(ilon)

end_time = timer()
print("Tijd voor bias correctie:", end_time - start_time)

<xarray.DataArray 'lat' ()> Size: 8B
array(49.72711456)
Coordinates:
    lat      float64 8B 49.73
    height   float64 8B 10.0
Attributes:
    axis:           Y
    standard_name:  latitude
    long_name:      Latitude
    units:          degrees_north
<xarray.DataArray 'lon' ()> Size: 8B
array(2.8125)
Coordinates:
    lon      float64 8B 2.812
    height   float64 8B 10.0
Attributes:
    axis:           X
    standard_name:  longitude
    long_name:      Longitude
    units:          degrees_east
<xarray.DataArray 'lat' ()> Size: 8B
array(49.72711456)
Coordinates:
    lat      float64 8B 49.73
    height   float64 8B 10.0
Attributes:
    axis:           Y
    standard_name:  latitude
    long_name:      Latitude
    units:          degrees_north
<xarray.DataArray 'lon' ()> Size: 8B
array(4.21875)
Coordinates:
    lon      float64 8B 4.219
    height   float64 8B 10.0
Attributes:
    axis:           X
    standard_name:  longitude
    long_name:      Longitude
    units:          degr

In [79]:
#De namen veranderen zodat de onderliggende functies gewoon kunnen blijven draaien
#da_corrected_ssp585_4 = da_corrected_ssp585_4.rename({"lat": "latitude", "lon": "longitude"})
#da_corrected_ssp585_4 = da_corrected_ssp585_4.rename({"time": "valid_time"})

df_corrected_ssp585_4 = da_corrected_ssp585_4.to_dataframe().reset_index()
df_corrected_ssp585_4

#df_corrected_ssp585.max() #Maximale waarde bias corrected CMIP6
bias_corr_ds_ssp585_4 = da_corrected_ssp585_4.to_dataset()

# Opslaan
bias_corr_ds_ssp585_4.to_netcdf('/Users/anne-sophiesimons/Documents/Data thesis/Bias Corrected data/België/bias_corr_ssp585_4')

In [80]:
bias_corr_ssp585_4 = xr.load_dataset('/Users/anne-sophiesimons/Documents/Data thesis/Bias Corrected data/België/bias_corr_ssp585_4')

In [81]:
bias_corr_ssp585_4

<xarray.Dataset> Size: 878kB
Dimensions:     (valid_time: 15673, latitude: 2, longitude: 3, member: 1)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 125kB 2015-01-01T12:00:00 ... 210...
  * latitude    (latitude) float64 16B 49.73 51.13
  * longitude   (longitude) float64 24B 2.812 4.219 5.625
  * member      (member) int64 8B 0
Data variables:
    historical  (valid_time, latitude, longitude, member) float64 752kB 15.3 ...

## Bias correction historical data

In [6]:
#Read all ERA5 hazard documents
#Load ERA5 data
ds2000_2003 = xr.open_dataset('/Users/anne-sophiesimons/Documents/Data thesis/Data België/wind_gust_00_03')
ds2004_2007 = xr.open_dataset('/Users/anne-sophiesimons/Documents/Data thesis/Data België/wind_gust_04_07')
ds2008_2011 = xr.open_dataset('/Users/anne-sophiesimons/Documents/Data thesis/Data België/wind_gust_08_11')
ds2012_2014 = xr.open_dataset('/Users/anne-sophiesimons/Documents/Data thesis/Data België/wind_gust_12_14')

ds_combined = xr.concat([ds2000_2003, ds2004_2007, ds2008_2011, ds2012_2014], dim="valid_time")

#Calculate the daily maxima WG10 from ERA5 at each coordinate
df_combined = ds_combined.to_dataframe().reset_index()
df_combined


df_daily_max = df_combined.groupby(
    [df_combined['valid_time'].dt.date, df_combined['latitude'], df_combined['longitude']]
)['fg10'].max().reset_index()

df_daily_max

#'valid_time' must be a datetime
df_daily_max['valid_time'] = pd.to_datetime(df_daily_max['valid_time'])

#DataFrame to xarray Dataset
ds_daily_max = xr.Dataset.from_dataframe(df_daily_max.set_index(['valid_time', 'latitude', 'longitude']))

In [7]:
#Load CMIP6 data
bncmip6_historical = xr.open_dataset('/Users/anne-sophiesimons/Documents/Data thesis/sfcWindmax_day_EARTH_historical_4.nc')

#Boundaries of Belgium
lat_min, lat_max = 49.5, 51.6
lon_min, lon_max = 2.5, 6.5

#Select the lat and lon of Belgium
ds_belgium_historical = bncmip6_historical.sel(
    lat=slice(lat_min, lat_max),
    lon=slice(lon_min, lon_max)
)

#Get winter months
winter_months = [1, 2, 3, 10, 11, 12]

cmip6_hist_winter = ds_belgium_historical.sel(
    time=ds_belgium_historical.time.dt.month.isin(winter_months)
)

#Filter time between 2000-01-01 and 2014-12-31
start_date = np.datetime64('2000-01-01')
end_date   = np.datetime64('2015-01-01')

ds_belgium_historical = cmip6_hist_winter.sel(
    time=slice(start_date, end_date)
)

#Set to dataframe
df = ds_belgium_historical.to_dataframe().reset_index()

import xarray as xr
import numpy as np
from timeit import default_timer as timer

#Open datasets
bncmip6_historical = ds_belgium_historical
fnera5 = ds_daily_max

#Parameters
savencdf = True
pastname = "historical"  

#ERA5 data
era_da = fnera5['fg10']
era_df = era_da.to_dataframe().reset_index()

#Get lat and lon from CMIP6 dataset
latout = bncmip6_historical.lat
lonout = bncmip6_historical.lon
#Get timesteps from CMIP6 dataset
daysid = bncmip6_historical.time
memid = bncmip6_historical.member if "member" in bncmip6_historical.dims else [0] 
memid
daysid
latout
lonout

<xarray.DataArray 'lon' (lon: 6)> Size: 48B
array([2.8125  , 3.515625, 4.21875 , 4.921875, 5.625   , 6.328125])
Coordinates:
  * lon      (lon) float64 48B 2.812 3.516 4.219 4.922 5.625 6.328
    height   float64 8B 10.0
Attributes:
    bounds:         lon_bnds
    units:          degrees_east
    axis:           X
    long_name:      Longitude
    standard_name:  longitude

In [8]:
#Load CMIP6 data
bncmip6_historical_all = xr.open_dataset('/Users/anne-sophiesimons/Documents/Data thesis/sfcWindmax_day_EARTH_historical_4.nc')
import xarray as xr

#Boundaries of Belgium
lat_min, lat_max = 49.5, 51.6
lon_min, lon_max = 2.5, 6.5

#Select only lon and lat of Belgium
ds_belgium_hist_all = bncmip6_historical_all.sel(
    lat=slice(lat_min, lat_max),
    lon=slice(lon_min, lon_max)
)

bncmip6_hist_all = ds_belgium_hist_all

#Select winter months
winter_months = [1, 2, 3, 10, 11, 12]

months_hist_all = ds_belgium_hist_all.sel(
    time=ds_belgium_hist_all.time.dt.month.isin(winter_months)
)

daysid_hist_all = pd.to_datetime(months_hist_all['time'].values)

da_corrected_hist_all = xr.DataArray(
    np.zeros([len(daysid_hist_all), latout.size, lonout.size, len(memid)]),
    coords=[daysid_hist_all, latout, lonout, memid],
    dims=['time', 'lat', 'lon', 'member'],
    name=pastname
)


df_corrected_hist_all = da_corrected_hist_all.to_dataframe().reset_index()
df_corrected_hist_all

,time,lat,lon,member,historical
0,1984-01-01 12:00:00,50.175308,2.812500,0,0.0
1,1984-01-01 12:00:00,50.175308,3.515625,0,0.0
2,1984-01-01 12:00:00,50.175308,4.218750,0,0.0
3,1984-01-01 12:00:00,50.175308,4.921875,0,0.0
4,1984-01-01 12:00:00,50.175308,5.625000,0,0.0
...,...,...,...,...,...
101695,2014-12-31 12:00:00,51.578810,3.515625,0,0.0
101696,2014-12-31 12:00:00,51.578810,4.218750,0,0.0
101697,2014-12-31 12:00:00,51.578810,4.921875,0,0.0
101698,2014-12-31 12:00:00,51.578810,5.625000,0,0.0


In [9]:
#Load CMIP6 data
bncmip6_historical_all = xr.open_dataset('/Users/anne-sophiesimons/Documents/Data thesis/sfcWindmax_day_EARTH_historical_4.nc')

#ERA5 data
era_da = fnera5['fg10']
era_df = era_da.to_dataframe().reset_index()

#Get lat and lon from CMIP6 dataset
latout = bncmip6_historical.lat
lonout = bncmip6_historical.lon
#Get timesteps of CMIP6 dataset
daysid = bncmip6_historical.time
memid = bncmip6_historical.member if "member" in bncmip6_historical.dims else [0]  # check of er leden zijn
memid
daysid
latout
lonout

#Interpolate ERA5 data
era_qt_rg = era_da.interp(latitude =latout, longitude =lonout, method='linear', kwargs={"fill_value": 'extrapolate'})
era_qt_rg

#Start bias correction
start_time = timer()

for ilat in latout:
    for ilon in lonout:
        #Select GCM data on the gridpoint
        ncdfw_point = bncmip6_historical.sel(lat=ilat, lon=ilon)
        ncdfw_df = bncmip6_historical.sel(lat=ilat, lon=ilon).to_dataframe().drop(['lat','lon'], axis=1).unstack()

        #Get oberservation of ERA5 on the gridpoint
        dat_obs = era_qt_rg.sel(lat=ilat, lon=ilon, method = 'nearest').to_dataframe().drop(['lat','lon'], axis=1)['fg10']
        dat_mod = bncmip6_historical.sel(lat=ilat, lon=ilon, method = 'nearest').to_dataframe().drop(['lat','lon'], axis=1)['sfcWindmax'].unstack()
        dat_mod_all = months_hist_all.sel(lat=ilat, lon=ilon, method = 'nearest').to_dataframe().drop(['lat','lon'], axis=1)['sfcWindmax'].unstack()
        dat_mod = dat_mod.iloc[:, 0]
        dat_mod_all = dat_mod_all.iloc[:, 0]
        dat_mod.index = dat_mod.index.normalize() 
        dat_mod_all.index = dat_mod_all.index.normalize() 

        #Get only the overlapping time periods between CMIP6 and ERA5 data
        dat_mod = dat_mod.reindex(dat_obs.index).dropna()

        #Bias correction
        dat_mod_corrected = bias_correct_time_series(dat_mod, dat_obs, dat_mod_all) 

        #Save  
        da_corrected_hist_all.loc[dict(lat=ilat, lon=ilon, member=0)] = dat_mod_corrected.values

        print(ilat)
        print(ilon)

end_time = timer()
print("Time for bias correction", end_time - start_time)

<xarray.DataArray 'lat' ()> Size: 8B
array(50.17530806)
Coordinates:
    lat      float64 8B 50.18
    height   float64 8B 10.0
Attributes:
    bounds:         lat_bnds
    units:          degrees_north
    axis:           Y
    long_name:      Latitude
    standard_name:  latitude
<xarray.DataArray 'lon' ()> Size: 8B
array(2.8125)
Coordinates:
    lon      float64 8B 2.812
    height   float64 8B 10.0
Attributes:
    bounds:         lon_bnds
    units:          degrees_east
    axis:           X
    long_name:      Longitude
    standard_name:  longitude
<xarray.DataArray 'lat' ()> Size: 8B
array(50.17530806)
Coordinates:
    lat      float64 8B 50.18
    height   float64 8B 10.0
Attributes:
    bounds:         lat_bnds
    units:          degrees_north
    axis:           Y
    long_name:      Latitude
    standard_name:  latitude
<xarray.DataArray 'lon' ()> Size: 8B
array(3.515625)
Coordinates:
    lon      float64 8B 3.516
    height   float64 8B 10.0
Attributes:
    bounds:       

In [10]:
#Change the names of the variables such that the underlying functions can run
da_corrected_hist_all = da_corrected_hist_all.rename({"lat": "latitude", "lon": "longitude"})
da_corrected_hist_all = da_corrected_hist_all.rename({"time": "valid_time"})

df_corrected_historical_all = da_corrected_hist_all.to_dataframe().reset_index()

bias_corr_ds_historical_all = da_corrected_hist_all.to_dataset()

#Save
#bias_corr_ds_historical_all.to_netcdf('/Users/anne-sophiesimons/Documents/Data thesis/Bias Corrected data/België/EARTH_bias_corr_historical_4')

## Bias Correction for future data (SSP585)

In [11]:
#Read all ERA5 hazard documents
#Load ERA5 data
ds2000_2003 = xr.open_dataset('/Users/anne-sophiesimons/Documents/Data thesis/Data België/wind_gust_00_03')
ds2004_2007 = xr.open_dataset('/Users/anne-sophiesimons/Documents/Data thesis/Data België/wind_gust_04_07')
ds2008_2011 = xr.open_dataset('/Users/anne-sophiesimons/Documents/Data thesis/Data België/wind_gust_08_11')
ds2012_2014 = xr.open_dataset('/Users/anne-sophiesimons/Documents/Data thesis/Data België/wind_gust_12_14')

ds_combined = xr.concat([ds2000_2003, ds2004_2007, ds2008_2011, ds2012_2014], dim="valid_time")

#Calculate the daily maxima WG10 from ERA5 at each coordinate
df_combined = ds_combined.to_dataframe().reset_index()
df_combined


df_daily_max = df_combined.groupby(
    [df_combined['valid_time'].dt.date, df_combined['latitude'], df_combined['longitude']]
)['fg10'].max().reset_index()

df_daily_max

#'valid_time' must be a datetime
df_daily_max['valid_time'] = pd.to_datetime(df_daily_max['valid_time'])

#DataFrame to xarray Dataset
ds_daily_max = xr.Dataset.from_dataframe(df_daily_max.set_index(['valid_time', 'latitude', 'longitude']))

In [12]:
#Load CMIP6 data
bncmip6_historical = xr.open_dataset('/Users/anne-sophiesimons/Documents/Data thesis/sfcWindmax_day_EARTH_historical_1.nc')
#Boundaries of Belgium
lat_min, lat_max = 49.5, 51.6
lon_min, lon_max = 2.5, 6.5

#Select the lat and lon of Belgium
ds_belgium_historical = bncmip6_historical.sel(
    lat=slice(lat_min, lat_max),
    lon=slice(lon_min, lon_max)
)

#Select winter months
winter_months = [1, 2, 3, 10, 11, 12]

cmip6_hist_winter = ds_belgium_historical.sel(
    time=ds_belgium_historical.time.dt.month.isin(winter_months)
)

#Filter time between 2000-01-01 and 2014-12-31
start_date = np.datetime64('2000-01-01')
end_date   = np.datetime64('2015-01-01')

ds_belgium_historical = cmip6_hist_winter.sel(
    time=slice(start_date, end_date)
)

#Set to dataframe
df = ds_belgium_historical.to_dataframe().reset_index()

import xarray as xr
import numpy as np
from timeit import default_timer as timer

#Open datasets
bncmip6_historical = ds_belgium_historical
fnera5 = ds_daily_max

#Parameters
savencdf = True
pastname = "historical"  
 
#ERA5 data
era_da = fnera5['fg10']
era_df = era_da.to_dataframe().reset_index()

#Get lat and lon of CMIP6 datasets
latout = bncmip6_historical.lat
lonout = bncmip6_historical.lon
#Get all timestaps of CMIP6 dataset
daysid = bncmip6_historical.time
memid = bncmip6_historical.member if "member" in bncmip6_historical.dims else [0]
memid
daysid
latout
lonout

<xarray.DataArray 'lon' (lon: 6)> Size: 48B
array([2.8125  , 3.515625, 4.21875 , 4.921875, 5.625   , 6.328125])
Coordinates:
  * lon      (lon) float64 48B 2.812 3.516 4.219 4.922 5.625 6.328
    height   float64 8B 10.0
Attributes:
    bounds:         lon_bnds
    units:          degrees_east
    axis:           X
    long_name:      Longitude
    standard_name:  longitude

In [13]:
#Load CMIP6 data under the scenario SSP 5855
bncmip6_ssp585 = xr.open_dataset('/Users/anne-sophiesimons/Documents/Data thesis/sfcWindmax_day_EARTH_SSP_SSP585_1.nc')
winter_months = [1, 2, 3, 10, 11, 12]
bncmip6_ssp585_winter = bncmip6_ssp585.sel(
    time=bncmip6_ssp585.time.dt.month.isin(winter_months)
)

#Boundaries of Belgium
lat_min, lat_max = 49.5, 51.6
lon_min, lon_max = 2.5, 6.5

#Select only lon and lat of Belgium
ds_belgium_ssp585 = bncmip6_ssp585_winter.sel(
    lat=slice(lat_min, lat_max),
    lon=slice(lon_min, lon_max)
)

bncmip6_ssp585 = ds_belgium_ssp585

#Get the winter months
winter_months = [1, 2, 3, 10, 11, 12]

months_ssp585 = ds_belgium_ssp585.sel(
    time=ds_belgium_ssp585.time.dt.month.isin(winter_months)
)

daysid_ssp585 = pd.to_datetime(months_ssp585['time'].values)

da_corrected_ssp585 = xr.DataArray(
    np.zeros([len(daysid_ssp585), latout.size, lonout.size, len(memid)]),
    coords=[daysid_ssp585, latout, lonout, memid],
    dims=['time', 'lat', 'lon', 'member'],
    name=pastname
)


df_corrected_ssp585 = da_corrected_ssp585.to_dataframe().reset_index()
df_corrected_ssp585

,time,lat,lon,member,historical
0,2070-01-01 12:00:00,50.175308,2.812500,0,0.0
1,2070-01-01 12:00:00,50.175308,3.515625,0,0.0
2,2070-01-01 12:00:00,50.175308,4.218750,0,0.0
3,2070-01-01 12:00:00,50.175308,4.921875,0,0.0
4,2070-01-01 12:00:00,50.175308,5.625000,0,0.0
...,...,...,...,...,...
101677,2100-12-31 12:00:00,51.578810,3.515625,0,0.0
101678,2100-12-31 12:00:00,51.578810,4.218750,0,0.0
101679,2100-12-31 12:00:00,51.578810,4.921875,0,0.0
101680,2100-12-31 12:00:00,51.578810,5.625000,0,0.0


In [15]:
#Interpolate ERA5 data
era_qt_rg = era_da.interp(latitude =latout, longitude =lonout, method='linear', kwargs={"fill_value": 'extrapolate'})
era_qt_rg

#Start bias correction
start_time = timer()

for ilat in latout:
    for ilon in lonout:
        #Select data on the gridpoint
        ncdfw_point = bncmip6_historical.sel(lat=ilat, lon=ilon)
        ncdfw_df = bncmip6_historical.sel(lat=ilat, lon=ilon).to_dataframe().drop(['lat','lon'], axis=1).unstack()

        #Get ERA5 observations on the gridpoint
        dat_obs = era_qt_rg.sel(lat=ilat, lon=ilon, method = 'nearest').to_dataframe().drop(['lat','lon'], axis=1)['fg10']
        dat_mod = bncmip6_historical.sel(lat=ilat, lon=ilon, method = 'nearest').to_dataframe().drop(['lat','lon'], axis=1)['sfcWindmax'].unstack()
        dat_mod_ssp585 = bncmip6_ssp585_winter.sel(lat=ilat, lon=ilon, method = 'nearest').to_dataframe().drop(['lat','lon'], axis=1)['sfcWindmax'].unstack()
        dat_mod = dat_mod.iloc[:, 0]
        dat_mod_ssp585 = dat_mod_ssp585.iloc[:, 0]
        dat_mod.index = dat_mod.index.normalize() 
        dat_mod_ssp585.index = dat_mod_ssp585.index.normalize() 


        #Get only the overlapping period between ERA5 and CMIP6 data
        dat_mod = dat_mod.reindex(dat_obs.index).dropna()

        #Bias correction
        dat_mod_corrected = bias_correct_time_series(dat_mod, dat_obs, dat_mod_ssp585) 

        da_corrected_ssp585.loc[dict(lat=ilat, lon=ilon, member=0)] = dat_mod_corrected.values

        print(ilat)
        print(ilon)

end_time = timer()
print("Time for bias correction:", end_time - start_time)

<xarray.DataArray 'lat' ()> Size: 8B
array(50.17530806)
Coordinates:
    lat      float64 8B 50.18
    height   float64 8B 10.0
Attributes:
    bounds:         lat_bnds
    units:          degrees_north
    axis:           Y
    long_name:      Latitude
    standard_name:  latitude
<xarray.DataArray 'lon' ()> Size: 8B
array(2.8125)
Coordinates:
    lon      float64 8B 2.812
    height   float64 8B 10.0
Attributes:
    bounds:         lon_bnds
    units:          degrees_east
    axis:           X
    long_name:      Longitude
    standard_name:  longitude
<xarray.DataArray 'lat' ()> Size: 8B
array(50.17530806)
Coordinates:
    lat      float64 8B 50.18
    height   float64 8B 10.0
Attributes:
    bounds:         lat_bnds
    units:          degrees_north
    axis:           Y
    long_name:      Latitude
    standard_name:  latitude
<xarray.DataArray 'lon' ()> Size: 8B
array(3.515625)
Coordinates:
    lon      float64 8B 3.516
    height   float64 8B 10.0
Attributes:
    bounds:       

In [23]:
#Change the names of the variables to run the underlying function
da_corrected_ssp585 = da_corrected_ssp585.rename({"lat": "latitude", "lon": "longitude"})
da_corrected_ssp585 = da_corrected_ssp585.rename({"time": "valid_time"})

df_corrected_ssp585 = da_corrected_ssp585.to_dataframe().reset_index()
df_corrected_ssp585

bias_corr_ds_ssp585 = da_corrected_ssp585.to_dataset()

#Save
#bias_corr_ds_ssp585.to_netcdf('/Users/anne-sophiesimons/Documents/Data thesis/Bias Corrected data/België/SSP585_EARTH_1')